# 1.3 Filter data 

## 1.3 Filter data 

remove missing weight and negative GA

In [ ]:
import pickle
with open("/mnt/home/qinqiu/thesis/data/590_patients.pkl", "rb") as f:
    df_all = pickle.load(f)

In [ ]:
import os
import pickle
import zlib
import pandas as pd

# ============================================================
# 0) Load merged dataframe
# ============================================================
pkl_path = "/mnt/home/qinqiu/thesis/data/590_patients.pkl"
with open(pkl_path, "rb") as f:
    df_all = pickle.load(f)

print("Loaded df_all:", df_all.shape)
print("Unique patients before:", df_all["ids__uid"].nunique())


# ============================================================
# 1) Column names
# ============================================================
id_col = "ids__uid"
ga_col = "feats__ga_w"

if ga_col not in df_all.columns:
    raise ValueError(f"Cannot find GA column '{ga_col}' in df_all.columns.")


# ============================================================
# 2) Filtering
#    Only drop rows with invalid GA
#    Do NOT drop rows with missing weight,
#    because weight will not be used as a feature anymore.
# ============================================================
df_patients_filtered = df_all[df_all[ga_col] >= 0].copy()


# ============================================================
# 3) Sanity checks
# ============================================================
n_before_patients = df_all[id_col].nunique()
n_after_patients = df_patients_filtered[id_col].nunique()

print("After filtering shape:", df_patients_filtered.shape)
print("Unique patients after filtering:", n_after_patients)
print("Dropped rows due to GA < 0:", df_all.shape[0] - df_patients_filtered.shape[0])
print("Patients before:", n_before_patients, "| Patients after:", n_after_patients)

# Optional: report weight missingness only, but do not filter by it
weight_candidates = [c for c in df_all.columns if "weight" in c.lower()]
if weight_candidates:
    for c in weight_candidates:
        print(f"Missing values in {c}:", df_patients_filtered[c].isna().sum())


# ============================================================
# 4) Save with new no-weight-safe name
# ============================================================
out_dir = "/mnt/home/qinqiu/thesis/data"
os.makedirs(out_dir, exist_ok=True)

prefix = "data_patients_filtered"

out_csv = os.path.join(out_dir, f"{prefix}.csv")
out_pkl = os.path.join(out_dir, f"{prefix}.pkl")
out_pklz = os.path.join(out_dir, f"{prefix}.pklz")

df_patients_filtered.to_csv(out_csv, index=False)

with open(out_pkl, "wb") as f:
    pickle.dump(df_patients_filtered, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(out_pklz, "wb") as f:
    f.write(
        zlib.compress(
            pickle.dumps(df_patients_filtered, protocol=pickle.HIGHEST_PROTOCOL)
        )
    )

print("Saved CSV :", out_csv)
print("Saved PKL :", out_pkl)
print("Saved PKLZ:", out_pklz)

Remove adverse events,>35 ga, and intubated patients

In [ ]:
# -*- coding: utf-8 -*-

from pathlib import Path
from typing import Optional
import pandas as pd


def collect_sick_uids_from_pkl(
    pkl_path: str,
    out_csv: Optional[str] = None,
) -> pd.Index:

    df = pd.read_pickle(pkl_path)

    required_cols = ["ids__uid", "target__los", "target__abdominal_nec"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in DataFrame: {missing}")

    # Make sure targets are numeric
    los = pd.to_numeric(df["target__los"], errors="coerce").fillna(0)
    nec = pd.to_numeric(df["target__abdominal_nec"], errors="coerce").fillna(0)

    tmp = df[["ids__uid"]].copy()
    tmp["target__los"] = los
    tmp["target__abdominal_nec"] = nec

    # Patient-level: ever sick
    per_uid_max = tmp.groupby("ids__uid")[["target__los", "target__abdominal_nec"]].max()

    ever_sick = (
        (per_uid_max["target__los"] == 1) |
        (per_uid_max["target__abdominal_nec"] == 1)
    )

    sick_uids = per_uid_max.index[ever_sick]

    if out_csv is not None:
        out_path = Path(out_csv)
        out_path.parent.mkdir(parents=True, exist_ok=True)
        pd.Series(sick_uids, name="ids__uid").to_csv(out_path, index=False)

    return sick_uids


if __name__ == "__main__":

    PKL_PATH = "/mnt/home/qinqiu/myproject/Combined_data/filtered_dataset.pkl"

    # ✅ Save here
    OUT_CSV = "/mnt/home/qinqiu/thesis/data/sick_uids_los_or_nec.csv"

    sick_uids = collect_sick_uids_from_pkl(PKL_PATH, OUT_CSV)

    print("Patients ever had LOS/NEC:", len(sick_uids))
    print(list(sick_uids))

In [ ]:
import os
import zlib
import pickle
import pandas as pd
import numpy as np

# ============================================================
# 0) Load filtered dataframe
# ============================================================
pkl_path = "/mnt/home/qinqiu/thesis/data/data_patients_filtered.pkl"

with open(pkl_path, "rb") as f:
    df_patients_filtered = pickle.load(f)

print("Loaded filtered df:", df_patients_filtered.shape)
print("Unique patients:", df_patients_filtered["ids__uid"].nunique())


# ============================================================
# 1) Column names
# ============================================================
id_col = "ids__uid"
ga_col = "feats__ga_w"
time_col = "timestamp"   # confirmed by you

df_patients_filtered[id_col] = df_patients_filtered[id_col].astype(str)
df_patients_filtered[time_col] = pd.to_datetime(df_patients_filtered[time_col], errors="coerce")

if df_patients_filtered[time_col].isna().all():
    raise ValueError(f"All values in {time_col} are NaT after parsing.")

# each row covers [timestamp, timestamp + 10 minutes]
df_patients_filtered["window_start"] = df_patients_filtered[time_col]
df_patients_filtered["window_end"] = df_patients_filtered[time_col] + pd.Timedelta(minutes=10)


# ============================================================
# 2) Remove patients with adverse events (LOS or abdominal NEC)
# ============================================================
sick_uids_csv = "/mnt/home/qinqiu/thesis/data/sick_uids_los_or_nec.csv"
sick_uids = pd.read_csv(sick_uids_csv)["ids__uid"].astype(str)

n_before = df_patients_filtered[id_col].nunique()
rows_before = len(df_patients_filtered)

df_patients_clean = df_patients_filtered[
    ~df_patients_filtered[id_col].isin(set(sick_uids))
].copy()

n_after = df_patients_clean[id_col].nunique()
rows_after = len(df_patients_clean)

print(f"[Remove adverse] Patients: {n_before} -> {n_after} (removed {n_before - n_after})")
print(f"[Remove adverse] Rows:     {rows_before} -> {rows_after} (removed {rows_before - rows_after})")


# ============================================================
# 3) Remove patients with GA > 35
# ============================================================
patient_ga = df_patients_clean.groupby(id_col)[ga_col].first()
uids_ga_over35 = patient_ga[patient_ga > 35].index.astype(str)

n_before_ga = df_patients_clean[id_col].nunique()
rows_before_ga = len(df_patients_clean)

df_patients_clean = df_patients_clean[
    ~df_patients_clean[id_col].isin(set(uids_ga_over35))
].copy()

n_after_ga = df_patients_clean[id_col].nunique()
rows_after_ga = len(df_patients_clean)

print(f"[Remove GA>35] Patients: {n_before_ga} -> {n_after_ga} (removed {n_before_ga - n_after_ga})")
print(f"[Remove GA>35] Rows:     {rows_before_ga} -> {rows_after_ga} (removed {rows_before_ga - rows_after_ga})")


# ============================================================
# 4) Remove fully intubated patients
#    ids_intubated.txt: remove whole patient
# ============================================================
intubated_txt = "/mnt/home/qinqiu/thesis/data/ids_intubated.txt"

with open(intubated_txt, "r") as f:
    intubated_uids = [line.strip() for line in f if line.strip()]

intubated_uids = set(map(str, intubated_uids))

n_before_intub_all = df_patients_clean[id_col].nunique()
rows_before_intub_all = len(df_patients_clean)

df_patients_clean = df_patients_clean[
    ~df_patients_clean[id_col].isin(intubated_uids)
].copy()

n_after_intub_all = df_patients_clean[id_col].nunique()
rows_after_intub_all = len(df_patients_clean)

print(f"[Remove full intubated patients] Patients: {n_before_intub_all} -> {n_after_intub_all} "
      f"(removed {n_before_intub_all - n_after_intub_all})")
print(f"[Remove full intubated patients] Rows:     {rows_before_intub_all} -> {rows_after_intub_all} "
      f"(removed {rows_before_intub_all - rows_after_intub_all})")


# ============================================================
# 5) Remove rows overlapping intubation periods
#    ids_intubated_with_timings.csv
# ============================================================
intubation_timing_csv = "/mnt/home/qinqiu/thesis/data/ids_intubated_with_timings.csv"
timing_df = pd.read_csv(intubation_timing_csv)

required_cols = ["patient_id", "intubation_start", "intubation_end"]
for col in required_cols:
    if col not in timing_df.columns:
        raise ValueError(f"Missing required column: {col}")

timing_df["patient_id"] = timing_df["patient_id"].astype(str)
timing_df["intubation_start"] = pd.to_datetime(timing_df["intubation_start"], errors="coerce")
timing_df["intubation_end"] = pd.to_datetime(timing_df["intubation_end"], errors="coerce")

n_timing_before = len(timing_df)
timing_df = timing_df.dropna(subset=["patient_id", "intubation_start", "intubation_end"]).copy()
n_timing_after = len(timing_df)

print(f"[Timing rows] Valid timing rows: {n_timing_before} -> {n_timing_after}")

# remove rows whose 10-min window overlaps intubation interval
remove_mask = pd.Series(False, index=df_patients_clean.index)

for _, row in timing_df.iterrows():
    pid = row["patient_id"]
    start = row["intubation_start"]
    end = row["intubation_end"]

    # overlap condition:
    # [window_start, window_end] overlaps [start, end]
    this_mask = (
        (df_patients_clean[id_col] == pid) &
        (df_patients_clean["window_end"] >= start) &
        (df_patients_clean["window_start"] <= end)
    )

    remove_mask = remove_mask | this_mask

removed_rows_df = df_patients_clean.loc[remove_mask, [id_col, "window_start", "window_end"]].copy()

rows_before_timing = len(df_patients_clean)
patients_before_timing = df_patients_clean[id_col].nunique()

df_patients_clean = df_patients_clean.loc[~remove_mask].copy()

rows_after_timing = len(df_patients_clean)
patients_after_timing = df_patients_clean[id_col].nunique()

print(f"[Remove intubation overlap] Rows:     {rows_before_timing} -> {rows_after_timing} "
      f"(removed {rows_before_timing - rows_after_timing})")
print(f"[Remove intubation overlap] Patients: {patients_before_timing} -> {patients_after_timing} "
      f"(removed {patients_before_timing - patients_after_timing})")

if len(removed_rows_df) > 0:
    print("\nTop patients by removed overlapping windows:")
    print(removed_rows_df.groupby(id_col).size().sort_values(ascending=False).head(20))


# ============================================================
# 6) Drop helper columns if you do not want to keep them
# ============================================================
df_patients_clean = df_patients_clean.drop(columns=["window_start", "window_end"])


# ============================================================
# 7) Save cleaned dataset
# ============================================================
out_dir = "/mnt/home/qinqiu/thesis/data"
os.makedirs(out_dir, exist_ok=True)

prefix = "data_patients_filtered__no_los_nec__ga_le_35__no_intubation"
out_csv  = os.path.join(out_dir, f"{prefix}.csv")
out_pkl  = os.path.join(out_dir, f"{prefix}.pkl")
out_pklz = os.path.join(out_dir, f"{prefix}.pklz")

df_patients_clean.to_csv(out_csv, index=False)

with open(out_pkl, "wb") as f:
    pickle.dump(df_patients_clean, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(out_pklz, "wb") as f:
    f.write(zlib.compress(pickle.dumps(df_patients_clean, protocol=pickle.HIGHEST_PROTOCOL)))

print("Saved CSV :", out_csv)
print("Saved PKL :", out_pkl)
print("Saved PKLZ:", out_pklz)
print("Final shape:", df_patients_clean.shape)
print("Final unique patients:", df_patients_clean[id_col].nunique())

In [ ]:
import os
import pickle
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# 1) Load dataframe
# ============================================================
pkl_path = "/mnt/home/qinqiu/thesis/data/data_patients_filtered__no_los_nec__ga_le_35__no_intubation.pkl"
pic_dir = "/mnt/home/qinqiu/thesis/pictures"
os.makedirs(pic_dir, exist_ok=True)

with open(pkl_path, "rb") as f:
    df = pickle.load(f)

print("Loaded df:", df.shape)
print("Unique patients:", df["ids__uid"].nunique())


# ============================================================
# 2) Column names
# ============================================================
id_col = "ids__uid"
ga_col = "feats__ga_w"
pma_col = "target__pma_w"


# ============================================================
# 3) Build patient-level GA and window-level PMA
# ============================================================
# GA: one value per patient
ga_patient = df.groupby(id_col)[ga_col].first().dropna()

# PMA: keep all windows
pma_window = df[pma_col].dropna()

print("\n========== Patient-level GA statistics ==========")
print(ga_patient.describe())

print("\n========== Window-level PMA statistics ==========")
print(pma_window.describe())

print(f"\nNumber of patients in GA distribution: {len(ga_patient)}")
print(f"Number of windows in PMA distribution: {len(pma_window)}")


# ============================================================
# 4) Save statistics
# ============================================================
ga_stats_path = os.path.join(pic_dir, "patient_level_ga_statistics.csv")
pma_stats_path = os.path.join(pic_dir, "window_level_pma_statistics.csv")

ga_patient.describe().to_csv(ga_stats_path, header=["value"])
pma_window.describe().to_csv(pma_stats_path, header=["value"])

print("Saved:", ga_stats_path)
print("Saved:", pma_stats_path)


# ============================================================
# 5) Plot: patient-level GA distribution
# ============================================================
plt.figure(figsize=(8, 5))
plt.hist(ga_patient, bins=30)
plt.xlabel("Gestational Age (weeks)")
plt.ylabel("Number of patients")
plt.title("Patient-level Distribution of Gestational Age")
plt.tight_layout()

ga_fig = os.path.join(pic_dir, "patient_level_distribution_ga.png")
plt.savefig(ga_fig, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", ga_fig)


# ============================================================
# 6) Plot: window-level PMA distribution
# ============================================================
plt.figure(figsize=(8, 5))
plt.hist(pma_window, bins=30)
plt.xlabel("Postmenstrual Age (weeks)")
plt.ylabel("Number of windows")
plt.title("Window-level Distribution of PMA")
plt.tight_layout()

pma_fig = os.path.join(pic_dir, "window_level_distribution_pma.png")
plt.savefig(pma_fig, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", pma_fig)

Print missing data

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import ListedColormap

# ============================================================
# 1) Load the merged dataframe
#    Each row corresponds to one time window of one patient
# ============================================================


with open("/mnt/home/qinqiu/thesis/data/data_patients_filtered.pkl", "rb") as f:
    df_all = pickle.load(f)



# Column name identifying patients
id_col = "ids__uid"

# Feature columns: everything except patient ID
feature_cols = [c for c in df_all.columns if c != id_col]

# Ensure rows of the same patient are contiguous
# (row order within each patient is assumed to be temporal order)
df_all = df_all.sort_values([id_col], kind="stable")


# ============================================================
# 2) Define colormap
#    0 = present (coloured)
#    1 = missing (white)
# ============================================================
cmap = ListedColormap(["red", "white"])


# ============================================================
# 3) Create multi-page PDF
#    One page = one patient
# ============================================================
pdf_path = "/mnt/home/qinqiu/thesis/data/missingness_per_patient_filtered.pdf"

with PdfPages(pdf_path) as pdf:
    for pid, df_p in df_all.groupby(id_col):

        # Build missingness matrix:
        # rows   -> features
        # columns -> time windows
        # value  -> 1 if missing, 0 if present
        miss = df_p[feature_cols].isna().to_numpy(dtype=int).T

        n_features, n_windows = miss.shape

        # Dynamically scale figure size for readability
        fig_w = max(8, min(20, n_windows / 20))
        fig_h = max(6, min(20, n_features / 3))

        plt.figure(figsize=(fig_w, fig_h))
        plt.imshow(
            miss,
            aspect="auto",
            interpolation="nearest",
            cmap=cmap
        )

        plt.title(f"Missingness heatmap (patient {pid})\nColored cells indicate missing values")
        plt.xlabel("Time window (row order)")
        plt.ylabel("Features")

        # Show feature names only if not too many
        if n_features <= 80:
            plt.yticks(range(n_features), feature_cols, fontsize=7)
        else:
            plt.yticks([])

        plt.tight_layout()
        pdf.savefig()
        plt.close()

print("PDF saved to:", pdf_path)

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import ListedColormap


# ============================================================
# 1) Load the merged dataframe
#    Each row corresponds to one time window of one patient
# ============================================================
with open("/mnt/home/qinqiu/thesis/data/data_patients_filtered.pkl", "rb") as f:
    df_all = pickle.load(f)


# ============================================================
# 2) Column settings
# ============================================================
id_col = "ids__uid"

# Try to automatically find PMA column
pma_candidates = [
    "target__pma_w",
    "target_pma_w",
    "pma_w",
    "PMA",
    "PMA_w",
    "feats__pma_w",
]

pma_col = None
for c in pma_candidates:
    if c in df_all.columns:
        pma_col = c
        break

if pma_col is None:
    raise ValueError(
        "Could not find PMA column. Please set pma_col manually. "
        f"Available columns include:\n{df_all.columns.tolist()[:50]}"
    )

print("Using PMA column:", pma_col)


# ============================================================
# 3) Ignore weight feature
# ============================================================
# Main repeated/current weight column to remove
weight_cols_to_ignore = [
    c for c in df_all.columns
    if c == "feats__weight"
    or c.lower() in ["weight", "feats_weight"]
    or c.lower().endswith("__weight")
]

print("Ignoring weight columns:")
print(weight_cols_to_ignore)


# ============================================================
# 4) Define feature columns
#    Exclude:
#    - patient ID
#    - PMA column used for x-axis
#    - repeated/current weight
# ============================================================
exclude_cols = set([id_col, pma_col] + weight_cols_to_ignore)

feature_cols = [
    c for c in df_all.columns
    if c not in exclude_cols
]

print("Number of features plotted:", len(feature_cols))


# ============================================================
# 5) Sort rows
#    Patient first, then PMA within patient
# ============================================================
df_all = df_all.sort_values([id_col, pma_col], kind="stable")


# ============================================================
# 6) Define colormap
#    0 = present
#    1 = missing
# ============================================================
cmap = ListedColormap(["red", "white"])


# ============================================================
# 7) Create multi-page PDF
#    One page = one patient
# ============================================================
pdf_path = "/mnt/home/qinqiu/thesis/data/missingness_per_patient_filtered_xaxis_pma_no_weight.pdf"

with PdfPages(pdf_path) as pdf:
    for pid, df_p in df_all.groupby(id_col, sort=False):

        df_p = df_p.sort_values(pma_col, kind="stable").reset_index(drop=True)

        # PMA values for x-axis
        pma_values = df_p[pma_col].to_numpy(dtype=float)

        # Build missingness matrix:
        # rows    -> features
        # columns -> time windows ordered by PMA
        # value   -> 1 if missing, 0 if present
        miss = df_p[feature_cols].isna().to_numpy(dtype=int).T

        n_features, n_windows = miss.shape

        if n_windows == 0:
            continue

        # Dynamically scale figure size for readability
        fig_w = max(8, min(22, n_windows / 20))
        fig_h = max(6, min(22, n_features / 3))

        plt.figure(figsize=(fig_w, fig_h))

        plt.imshow(
            miss,
            aspect="auto",
            interpolation="nearest",
            cmap=cmap
        )

        plt.title(
            f"Missingness heatmap (patient {pid})\n"
            f"x-axis = PMA, weight excluded"
        )
        plt.xlabel("PMA (weeks)")
        plt.ylabel("Features")

        # ----------------------------------------------------
        # X-axis ticks: show PMA instead of window index
        # ----------------------------------------------------
        max_xticks = 10

        if n_windows <= max_xticks:
            tick_positions = np.arange(n_windows)
        else:
            tick_positions = np.linspace(0, n_windows - 1, max_xticks).astype(int)

        tick_labels = [f"{pma_values[i]:.1f}" for i in tick_positions]

        plt.xticks(
            tick_positions,
            tick_labels,
            rotation=45,
            ha="right",
            fontsize=8,
        )

        # Show feature names only if not too many
        if n_features <= 80:
            plt.yticks(range(n_features), feature_cols, fontsize=7)
        else:
            plt.yticks([])

        plt.tight_layout()
        pdf.savefig()
        plt.close()

print("PDF saved to:", pdf_path)